In [ ]:
"""
Live system-audio transcription (Windows WASAPI loopback + faster-whisper).

Captures whatever is playing through your default speakers -- e.g. a YouTube ad
in the browser -- and prints a rolling transcript.

Setup:
    pip install pyaudiowpatch faster-whisper scipy numpy

"""
# %pip install numpy scipy pyaudiowpatch faster-whisper

import numpy as np
import scipy.signal as sps
import pyaudiowpatch as pyaudio
from faster_whisper import WhisperModel

MODEL         = "distil-large-v3"   # english-only; use "large-v3-turbo" for other langs
COMPUTE       = "int8"              # int8 = fast on CPU; "int8_float16" if you move to GPU
TARGET_SR     = 16000               # Whisper expects 16 kHz mono
CHUNK_SECONDS = 5                   # transcribe every N seconds of buffered audio
LANG          = "en"               # set None to auto-detect

model = WhisperModel(MODEL, device="cpu", compute_type=COMPUTE)
pa = pyaudio.PyAudio()

# --- find the loopback device for the current default speakers ---
wasapi      = pa.get_host_api_info_by_type(pyaudio.paWASAPI)
default_out = pa.get_device_info_by_index(wasapi["defaultOutputDevice"])
loop = next(d for d in pa.get_loopback_device_info_generator()
            if default_out["name"] in d["name"])

src_sr   = int(loop["defaultSampleRate"])      # usually 48000
channels = int(loop["maxInputChannels"])       # usually 2
frames_per_chunk = src_sr * CHUNK_SECONDS

stream = pa.open(format=pyaudio.paInt16, channels=channels, rate=src_sr,
                 input=True, input_device_index=loop["index"],
                 frames_per_buffer=1024)

print(f"capturing '{loop['name']}' @ {src_sr} Hz / {channels}ch "
      f"-> transcribing every {CHUNK_SECONDS}s  (Ctrl-C to stop)\n")

buf = np.empty((0,), dtype=np.float32)
try:
    while True:
        raw = stream.read(1024, exception_on_overflow=False)
        a = np.frombuffer(raw, np.int16).astype(np.float32) / 32768.0
        if channels > 1:
            a = a.reshape(-1, channels).mean(axis=1)            # downmix to mono
        buf = np.concatenate([buf, a])

        if len(buf) >= frames_per_chunk:
            mono16 = sps.resample_poly(buf, TARGET_SR, src_sr)  # e.g. 48k -> 16k
            buf = np.empty((0,), dtype=np.float32)
            segs, _ = model.transcribe(
                mono16.astype(np.float32),
                language=LANG,
                vad_filter=True,                                # kills music hallucination
                beam_size=5,
            )
            text = " ".join(s.text.strip() for s in segs).strip()
            if text:
                print(text)
except KeyboardInterrupt:
    pass
finally:
    stream.stop_stream()
    stream.close()
    pa.terminate()

c:\Users\chris\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


capturing '3 - 25M2S (2- AMD High Definition Audio Device) [Loopback]' @ 48000 Hz / 2ch -> transcribing every 5s  (Ctrl-C to stop)

People can be on a mint family plan and get our lowest price of just $15 a month per person.
